# What is Azure Data Factory

**Azure Data Factory (ADF)** is a cloud-based **ETL (Extract, Transform, Load)** and data integration service. It serves as the backbone for *Azure* data engineering solutions, designed to help you orchestrate, migrate, and transform data between various sources and destinations.

Key features include:
* **Data Migration:** It allows you to move data from diverse sources—such as *SQL databases*, *CSV files*, or *APIs*—to destinations like *Azure Data Lake* or other cloud applications.
* **Connectors:** ADF provides a wide range of built-in connectors, making it easy to link with different data platforms seamlessly.
* **Data Flows:** This feature allows for visual data transformation using *Spark* clusters in the background, without requiring you to write code.
* **Orchestration:** You can build complex, automated pipelines that handle data movement and transformation according to specific business requirements.

# Resource Group
In **Azure**, a **Resource Group** acts as a logical container for your resources. You can think of it as a **folder** on your computer where you store files; similarly, in Azure, any service you create—such as *Azure Data Factory*, *Azure Data Lake*, or *Azure Synapse Analytics*—must be placed within a Resource Group.

Key points about Resource Groups:
* **Organization:** They allow you to group related resources together, making them easier to manage, deploy, and delete.
* **Mandatory:** You cannot create any resource in Azure without first associating it with a Resource Group.
* **Lifecycle Management:** Because resources are stored together, you can perform administrative tasks on the entire group, such as setting permissions or managing costs, which is highly efficient for data engineering projects.

# Azure Data lake and Blob storage
The key difference between **Azure Blob Storage** and **Azure Data Lake (Gen2)** lies in how they organize and manage your data:

* **Blob Storage:** Primarily relies on a **flat structure** using **containers** to store files. It is standard for general-purpose object storage.
* **Azure Data Lake:** Built on top of storage accounts, it introduces **hierarchical namespaces**. This allows you to create a **folder-within-folder structure**, which is essential for organizing large datasets in a way that mimics a real file system.

Essentially, by enabling the **hierarchical namespace** setting during the storage account creation, you transform a standard blob storage environment into a powerful **Data Lake**.

# Redundancy
In cloud computing, **data redundancy** is the practice of creating and maintaining multiple copies of your data across different physical locations or systems. It is a critical strategy for ensuring **data availability** and **durability**.

### Why Redundancy Matters:
* **Disaster Recovery:** If one storage location or server fails (due to hardware issues or local outages), your data remains accessible from another copy.
* **Increased Reliability:** By duplicating data, cloud providers ensure that your information is safe even if a specific data center or zone experiences a problem.
* **Continuous Operations:** It minimizes the risk of permanent data loss, acting as a safeguard for your most important business information.

In the context of *Azure*, there are several redundancy options available for storage accounts:
* **LRS (Locally Redundant Storage):** Data is duplicated within the same data center. This is the most cost-effective but has the lowest protection against regional failures.
* **ZRS (Zone Redundant Storage):** Data is replicated across different availability zones within a region.
* **GRS (Geo-Redundant Storage):** Provides protection across different geographical regions by copying data to a secondary region.
* **GZRS:** Combines zone redundancy with geo-redundancy for higher availability.

# Linked Serivce and DataSet

In *Azure Data Factory*, **Linked Services** and **Datasets** are fundamental concepts for managing data movement. They function as a hierarchy to establish connections and identify specific data structures.

### **Linked Services (The Connection)**
Think of a **Linked Service** as the **connection string** to an external service or data store. It acts as the bridge that allows *Azure Data Factory* to communicate with your source or destination, such as an *Azure SQL Database*, *Azure Data Lake*, or an external *API*.

### **Datasets (The Data Structure)**
Once a connection is established via a *Linked Service*, you need to define the specific data you want to work with. A **Dataset** represents the actual data structure within that connected service. 
* For example, if your *Linked Service* connects to a SQL Database, your **Dataset** would be the specific table or view you want to pull data from.
* In file-based storage, such as *Azure Data Lake*, a **Dataset** points to a specific container, folder, or file that you intend to ingest or write to.

**In summary:**
* **Linked Service** = Where is the data stored? (The connection)
* **Dataset** = Which specific table, folder, or file do I need? (The definition)

![image_1788319774952.png](./image_1788319774952.png "image_1788319774952.png")

# Copy Activity
The **Copy Activity** is a fundamental component of *Azure Data Factory* used to migrate data from one location to another. In nearly 99.99% of data solutions, this activity is essential for moving data between sources and destinations, which are referred to as 'syncs' within *Azure Data Factory*.

To function effectively, a **Copy Activity** requires two primary configurations:
* **Connection (Linked Service):** This establishes the bridge between *Azure Data Factory* and your data source or destination.
* **Dataset:** This defines the actual data structure within those connected services.

# Scenario 1

A company that currently stores large volumes of **structured and semi-structured data** in an **on-premise data warehouse** and local file servers. The challenge is migrating this data into *Azure* for cloud-based analytics.

In this video, the creator addresses the topic of **Integration Runtime (IR)** in *Azure Data Factory* through the following interview question:

**Question:** "What are the different types of integration runtimes (IR) in ADF, and when should you use each one?"

**Explanation and Solution:**
The creator explains that an *Integration Runtime* is the compute engine providing the computational resources necessary for data movement, transformation, and orchestration. There are three main types:

*   **Azure IR:** This is the most common type (used 90% of the time). You should use *Azure IR* when both the source and the destination data reside within the cloud.
*   **Self-hosted IR:** This is required when dealing with on-premise data (e.g., *SQL Server*, local file systems, or hard drives). It acts as a bridge to pull data from on-premise environments into the cloud.
*   **Azure SSIS IR:** This is specifically designed for "lift and shift" operations to move existing *Microsoft SSIS* packages—a legacy ETL framework—into the cloud environment.

Additionally, the creator notes that you can manage these runtimes under the **Manage** tab in *ADF* and mentions that *Airflow* is now also supported for running existing DAGs.

### The Solution


To perform this migration, the following approach is recommended:

*   **Use Azure Data Factory (ADF):** ADF serves as the central orchestration tool for this ETL (Extract, Transform, Load) process.
*   **Self-Hosted Integration Runtime (SHIR):** Since the data resides on-premises, a standard cloud-based runtime cannot directly access your local servers. You must install a **Self-Hosted Integration Runtime** on a machine within your on-premises network to act as a bridge.
*   **Linked Services:** In ADF, you need to create a **Linked Service** for your on-premise data source (like a SQL Server) and configure it to connect via the SHIR you just created, rather than the default "AutoResolve" runtime.

### Expert Tips
*   **Scaling and Reliability:** For production environments, you should **never use your local laptop** as the host for the SHIR. Instead, deploy a **Virtual Machine (VM)** within your corporate network to host the SHIR to ensure it remains available for other developers and scheduled pipelines.
*   **Security:** By using the SHIR, your data remains secure behind your corporate firewall, as the SHIR makes outbound calls to *Azure*, eliminating the need to open inbound ports on your local network.

# Scenario 2

* **Scenario:** "Your company receives daily exchange rate data from a REST API. You need to ingest this data into an Azure Data Lake Storage Gen2 container on a daily basis."
* **Question:** "How would you design the pipeline in Azure Data Factory to ingest this REST API data, and how would you handle the authentication and dynamic configuration of the source URL?"

**The Solution Strategy:**
1. **Web Activity:** Use a *Web activity* to call the REST API, handle authentication (such as headers or tokens), and capture the API response.
2. **Pipeline Parameters:** To make the process dynamic, avoid hardcoding the URL. Instead, use **Pipeline Parameters** so the base URL and query parameters can be changed easily without modifying the activity configuration.
3. **Copy Activity with Dynamic Dataset:** Use a *Copy Activity* to move the captured data from the *Web activity* output to *Azure Data Lake Storage*. It's recommended to use **JSON** as the standard format for API ingestion but notes that if the API returns comma-separated data, an **HTTP Connector** is a better, more direct approach.
4. **Mapping:** By using the **Auto-mapping** feature in the *Copy Activity*, you can ingest hierarchical JSON data structures directly into the data lake without manual column mapping for every schema change.

**Key Explanation:**
* **Why Web Activity?** It is essential for interacting with REST API services where you need to pass specific headers or perform authenticated GET/POST requests that the standard *Copy Activity* source cannot handle alone.
* **Dynamic Sinks:** The solution allows for dynamic path naming, enabling you to store files in folders organized by date (e.g., `year/month/day`), which is a best practice for partitioning large datasets in *Azure Data Lake*.

# Scenario 3

**Scenario 3: Data Loading to Azure Data Lake using HTTP**. 

In this scenario, the company needs to ingest data from an external source using an **HTTP connector**. When data is received as **delimited text** (like CSV) directly through API calls, it is highly recommended to use an **HTTP connection** rather than a complex web activity, as it allows you to directly define the format and schema for the ingestion process. 

Key takeaways from this scenario include:
* **Efficiency:** Using the HTTP connector provides a more direct path for ingesting flat files compared to the standard Web activity approach.
* **Schema Handling:** Since CSV files lack an inherent schema, you can configure the pipeline to treat the first row of the data as the header/schema, ensuring the data is parsed correctly as it is moved into *Azure Data Lake Storage*.

# Scenario 4

*   **Scenario:** You are working on an *ELT pipeline* that includes an API call to fetch data from a third-party service. This specific API is known to fail occasionally due to temporary network glitches or rate-limiting issues.
*   **Question:** As a data engineer, how would you design the pipeline activity to handle these **intermittent failures** and ensure your pipelines remain robust?


### **The Solution & Explanation:**
To handle intermittent failures caused by network glitches or rate-limiting during an activity (like an API call), implement a **Retry Policy** within the pipeline settings.

1.  **Retry Settings:** Navigate to the activity settings in your *Azure Data Factory* pipeline. You will find a **Retry** field where you can specify the number of attempts the system should make if the activity fails.
2.  **Retry Interval:** You can also configure the **Retry interval** (in seconds), which dictates the waiting period between consecutive attempts. This is the standard, efficient way to make pipelines robust without requiring complex custom error handling.
3.  **Proactive Monitoring:** Beyond the automated retry, use the **Monitor** tab in the pipeline to investigate specific error messages if a pipeline fails permanently after all retries are exhausted.

# Scenario 5 

**In an organizational setup, you might be managing hundreds or even 200+ ETL pipelines. When a failure occurs, it can be extremely hectic and time-consuming to manually dig through every single pipeline to identify which one failed and why the data wasn't updated.**

**The question is:**
How would you design and implement a **monitoring and alerting mechanism** in *Azure Data Factory* (or *Synapse Analytics*) to notify the team immediately in case of any ETL pipeline failure, including critical details like the pipeline name, resource name, and environment?

To address the challenge of tracking failures across hundreds of *Azure Data Factory* pipelines, here is how to set up an automated **monitoring and alerting mechanism** directly within the platform.

**Steps to implement monitoring and alerts:**

1. **Navigate to the Monitor tab:** Within the *Azure Data Factory* studio, move from the management section to the **Monitor** tab.
2. **Access Alerts and Metrics:** Select the **Alerts and metrics** feature. This built-in functionality replaces older, more complex methods of building custom logic apps to trigger notifications.
3. **Create a new alert rule:** Click on **New alert rule** to begin configuration.
    * **Define Severity:** Assign a severity level based on the pipeline's business importance. For critical pipelines, set the **Severity** to 0 for immediate action, while lower-priority tasks can use a higher severity level.
    * **Target Criteria:** Select the appropriate metric for failure detection. It's recommended to use **Failed pipeline run metric** to monitor the entire pipeline, rather than just a single activity.
    * **Define Failure Types:** When configuring the alert, select all failure categories—**User error**, **System error**, and **Bad gateway**—to ensure you capture all possible causes.
    * **Configure Alert Logic:** Set the threshold (e.g., if failures are **greater than 1**) to account for intermittent network issues where a retry policy might successfully recover the pipeline.
    * **Frequency:** Set the evaluation frequency (e.g., every 1 minute) to ensure the system is alerted promptly if an issue persists.

# Scenario 6
**The Scenario:** You are working on a distributed computing project and are asked to optimize memory management and performance. The interviewer specifically brings up the fundamental memory architecture of *Spark*.

**The Question:** Explain the difference between **caching** and **persisting** data frames in *PySpark*, and describe how *Spark* splits memory between **storage** and **execution** to optimize performance.

Here are the differences between **caching** and **persisting** in *PySpark* as part of performance and memory management discussion.

### **Key Concepts Explained:**

*   **Caching vs. Persisting:** 
    *   **Cache:** This is a convenience method that uses default storage levels (`MEMORY_ONLY`). It is essentially a shortcut for common use cases where you want to keep the data frame in memory for repeated access.
    *   **Persist:** This provides more flexibility, allowing you to specify different **storage levels**. You can choose to store data in memory only, on disk, or a combination of both (`MEMORY_AND_DISK`), and whether to serialize the data (which saves space at the cost of higher CPU usage).

*   **Memory Split (Execution vs. Storage):**
    *   *Spark* divides the heap memory of an executor into two main regions: **Storage memory** (used for caching RDDs/DataFrames) and **Execution memory** (used for computations like joins, shuffles, and sorts).
    *   A key aspect of *Spark's* architecture is that this boundary is **dynamic**. *Spark* can borrow memory from one region for the other if needed, which prevents memory wastage and optimizes performance based on the specific workload.

*   **Practical Takeaway:** 
    *   Use `persist` when you have specific requirements regarding how much data you can fit in memory or if the data frame exceeds available memory and needs to spill to disk. 
    *   Avoid unnecessary caching, as it can lead to memory overhead and potential garbage collection issues if not managed correctly.

# Scenario 7

**The Scenario:** You are working on an *Azure Data Factory* pipeline that requires calling a *REST API* via a web activity. This API requires an **access key** for authentication. 

**The Question:** How can you securely manage and retrieve this sensitive access key using *Azure Key Vault* instead of hardcoding it within the pipeline configuration?

For **Scenario-7: Azure Key Vaults with Azure Data Factory**, here is a secure end-to-end implementation process to avoid hardcoding secrets:

1. **Create the Azure Key Vault:** Deploy a new *Azure Key Vault* resource in your environment.
2. **Store the Secret:** Navigate to the **Secrets** tab within the *Key Vault*, create a new secret, and paste the sensitive access key as the value.
3. **Assign Access Policies:** To enable *Azure Data Factory* to read the secret, you must provide it with access. Go to **Access Control (IAM)** on the *Key Vault* resource and assign the **Key Vault Administrator** (or Key Vault Reader) role to the *Azure Data Factory* instance using its **System-Assigned Managed Identity**.
4. **Configure the Web Activity:** 
    * In your *Azure Data Factory* pipeline, use a **Web Activity** to call the *Key Vault* secret URL.
    * Use the **GET** method and set authentication to **System-assigned managed identity**.
5. **Secure Output Handling:** This is the most critical step to prevent security leaks. When configuring the web activity, you must **enable "Secure Output"** in the activity settings. This ensures that the secret retrieved from the *Key Vault* is not exposed in the plaintext pipeline execution logs.

**bonus interview questions** to test a candidate's practical, real-world knowledge beyond basic implementation:

1. **What are the necessary prerequisites before creating a secret?**
   You cannot simply start by creating a secret; you must first ensure you have the correct **permissions**. You must be assigned an appropriate role, such as *Key Vault Administrator*, *Key Vault Owner*, or a similar administrative role, via **Access Control (IAM)** to have the authority to manage secrets within the vault.

2. **How does the Azure Data Factory gain access to read secrets?**
   *Azure Data Factory* must have its own permissions to securely fetch information. This is accomplished by assigning a role (like *Key Vault Administrator* or *Reader*) to the data factory's **System-Assigned Managed Identity**. This ensures the data factory itself is authorized to access the *Key Vault* securely without requiring hardcoded credentials.

# Scenario 8

**Scenario-8: Triggers in Azure**, focusing on an alternative design approach for triggering pipelines based on file arrival.

### **The Scenario:**
An interviewer asks how you would design a solution to automatically trigger a pipeline whenever a new file is added to a specific folder in *Azure Data Lake Storage Gen2*, specifically **without** using an event-based trigger.

### **The Solution & Explanation:**
To achieve this without an event-based trigger, use a **Validation Activity**:

*   **How it works:** You place a *Validation Activity* at the beginning of your pipeline. This activity is configured to continuously search for the specific file in your storage container.
*   **Behavior:** The pipeline will stay on hold, remaining in a pending state at this specific step, while it polls the folder for the arrival of the file.
*   **Execution:** Once the *Validation Activity* detects that the file has arrived, it successfully completes, and the pipeline automatically proceeds to execute all subsequent activities in the sequence.
*   **Note:** This is not necessarily the "ideal" or most efficient way compared to event-based triggers, but it serves as a valid design pattern to keep a process waiting until a specific file is ready for processing.


# Storage Event

**Storage Event Trigger** (also known as an event-based trigger) is a feature in **Azure Data Factory** designed to automatically start a pipeline the moment a file is added to or deleted from a specific folder in your **Azure Data Lake Storage Gen2** account.

To implement a **Storage Event Trigger**, follow these steps:

* **Navigate to Triggers:** Open your pipeline in the Data Factory studio, click on **Add trigger**, and select **New**. 
* **Configure Type:** Set the trigger type to **Storage events**. 
* **Specify Location:** Select your Azure subscription and the specific **Storage account**. 
* **Define Path:** Input the **Container name** and the specific **folder path** within that container that the system should monitor. 
* **Set Conditions (Optional):** You can configure the trigger to filter by specific file paths, such as requiring the blob path to begin with a certain folder structure or file name (e.g., '2018/April/'). 
* **Select Events:** Choose whether the pipeline should fire on **Blob created** or **Blob deleted** events.

# Scenario 9
The business requires the ability to run ad-hoc SQL queries on this data directly from Azure Synapse Analytics without the overhead of copying the data into a dedicated Synapse SQL pool.

### **The Solution: Polybase and Serverless SQL**
The creator explains that the most effective way to handle this requirement is by using **Synapse Serverless SQL pools** combined with **Polybase** technology. This approach allows you to query external data sitting directly in your *Data Lake* as if it were a local table.

### **The Five-Step Implementation Process**
To set up this architecture, the creator outlines these specific steps:
1. **Create a Master Key:** Establish a *Master Key* for your database to secure the connection.
2. **Create a Database Credential:** Define a *Credential* that grants your *Synapse* workspace the necessary permissions to access the storage account (typically utilizing a Managed Identity).
3. **Create an External Data Source:** Define an *External Data Source* that points to the specific location of your data within the *Data Lake*.
4. **Create an External File Format:** Specify the format of the data (in this case, CSV) so *Synapse* knows how to parse the incoming files.
5. **Create an External Table:** Finally, create the *External Table* that maps your SQL schema to the underlying files in the *Data Lake*.

By following this workflow, your business users can perform SQL analysis on raw data immediately without needing to perform ETL processes to move that data into the dedicated warehouse.

# Scenario 10

### **The Scenario:**
You are tasked with building a pipeline to incrementally load data from a *SQL* source into an *Azure Data Lake* (e.g., in *Parquet* format). The business requirement is to avoid processing the entire dataset during every run, making the solution both **efficient** and **cost-effective**.

### **The Solution & Explanation:**

### **1. Change Data Capture (CDC)**
*   **How it works:** This is a modern, largely code-free feature in *Azure Data Factory* designed to automatically track data modifications at the source level.
*   **Best Use Case:** It works seamlessly with *SQL* databases. You must first enable *CDC* at both the database and table levels in your source system, after which you can connect it to *ADF* to capture changes automatically.
*   **Pros:** It is highly automated and reduces the need for custom logic, making it very efficient for *SQL* workloads.

### **2. Watermarking Approach**
*   **How it works:** This is considered the "backbone" of incremental loading. It relies on a specific column (e.g., `UpdateDate` or `CreateDate`) to track the most recently processed data.
*   **The Logic:** You use a *Copy Activity* with a `WHERE` clause (e.g., `WHERE UpdateDate > 'LastWatermarkValue'`) to fetch only new or updated records.
*   **The Workflow:**
    *   **Lookup Activity:** You perform a *Lookup* to get the `LastWatermarkValue` from your destination, which is then passed to the *Copy Activity* variable.
    *   **Updating the Marker:** After the load, a subsequent *Lookup* determines the new `MaxDate` from the source, which is then stored in a table or file to be used as the starting point for the next pipeline run.
*   **Modern Adaptation:** While historically done via stored procedures, the modern approach often involves writing the `MaxDate` to a small configuration file in the *Data Lake* to keep the process decoupled and flexible.

# Scenario 11

**The Scenario:**
In a distributed computing environment like *Spark*, you have a job processing large datasets that involves heavy **shuffling** operations (such as joins or aggregations). You are encountering performance bottlenecks where some tasks take significantly longer than others, leading to inefficient resource utilization and slow job completion times (data skew or suboptimal shuffle partitions).

**The Solution & Explanation:**
The creator explains that **AQE** is an optimization framework in *Spark* that allows the engine to re-optimize query plans **at runtime** based on statistics collected from finished stages. It performs three main optimizations:
1. **Dynamically coalescing shuffle partitions:** It automatically merges small shuffle partitions into larger ones to avoid the overhead of processing many tiny partitions.
2. **Dynamically switching join strategies:** It can convert a sort-merge join into a broadcast join if it discovers at runtime that one side of the join is small enough to fit in memory.
3. **Optimizing skewed joins:** It detects if one partition is significantly larger than others due to data skew and splits that skewed partition into smaller, more manageable sub-partitions to balance the workload across executors.

# Scenario 12

### **The Scenario**
You are tasked with building a **dynamic solution** that reads a schema at runtime and automatically maps source columns to destination columns. In a real-world environment, you often need to enforce **strict schema rules** to ensure data integrity, but you cannot rely on hardcoded mappings due to the high volume of files with varying structures. The objective is to achieve this **dynamic schema mapping** efficiently without manually defining columns for every new data source.

The explanation for **Scenario-13: Dynamic Mapping** in *Azure Data Factory* focuses on creating a **flexible, code-free ETL pipeline** that can handle shifting data structures without manual configuration for every single source file.

### **The Core Challenge**
In modern data engineering, you often deal with a high volume of files with varying structures. Manually mapping source columns to destination columns for each file is not scalable and requires constant maintenance whenever schemas evolve.

### **The Step-by-Step Solution**
The creator demonstrates a technique to dynamically inject mapping logic at runtime:

1. **Schema Capture:** First, you use the *Import Schemas* feature within the *Copy Activity* to generate the initial, correct structure for your data. This provides the base mapping that the system expects.
2. **Extracting the Translator Logic:** Once the mapping is set, you switch to the **Code** view of the *Copy Activity*. Here, you identify the `translator` dictionary, which contains the specific JSON logic that dictates how source fields align with sink columns.
3. **Implementing Dynamic Parameters:** You remove the hardcoded mapping from the UI and replace it with a **pipeline parameter** of type *Object*. By pasting the JSON structure you extracted into this parameter, the pipeline becomes "data-driven."
4. **Scalability and Flexibility:** Because the mapping is now a parameter, you can store different mapping configurations as JSON objects or arrays and pass them into the pipeline dynamically. This allows the same pipeline to handle diverse file structures automatically, solving the problem of schema evolution across thousands of files without needing to write custom code or manually update the activity for every new file arrival.

# Scenario 13

**The Scenario
You are tasked with processing 10 gigabytes of data using Spark. The goal is to determine the optimal number of executors, the memory requirements for each executor, and the number of cores needed to achieve maximum parallelism for the job.**

### **The Step-by-Step Calculation:**

1. **Calculate Total Partitions:**
   * The default *Spark* partition size is **128MB**. 
   * Convert the total data volume (10GB) into megabytes: $10 \times 1024 = 10,240$ MB.
   * Divide the total volume by the partition size ($10,240 \div 128$) to get approximately **80 partitions**.

2. **Determine Task Memory:**
   * If you assign **5 cores** per executor, each core processes one partition of 128MB simultaneously.
   * This requires a minimum of **640MB** of *Execution Memory* per executor to handle those 5 concurrent tasks.

3. **Account for Memory Overhead:**
   * The creator explains that *Execution Memory* typically accounts for only a portion (about 30%) of the total executor memory due to the internal *Spark* pool structure (which splits memory between *Storage* for caching and *Execution* for computation).
   * To arrive at the correct **Total Executor Memory**, multiply the required task memory (640MB) by a factor of **3.5 to 4**.
   * This results in a requirement of roughly **2.4GB to 2.6GB** of total memory per executor.

### **Key Takeaway:**
Following this methodology demonstrates an understanding of how *Spark* manages resources under the hood. By calculating the total data volume and accounting for the specific memory segments within an executor, you ensure your cluster has enough headroom to handle tasks without crashing.

# "How would you handle schema drift in an Azure Data Factory when dealing with evolving data sets?"

**Explanation and Solution:**
The creator explains that *Schema Drift* occurs when your source data changes—for example, when a new column is added to the source that does not yet exist in your destination.

*   **The Approach:** To handle this, you should use **Mapping Data Flows** within *ADF*.
*   **Configuration:** When creating your source data set in a mapping data flow, you are presented with two primary options: **Validate Schema** and **Allow Schema Drift**.
*   **The Mechanism:** By enabling the **Allow Schema Drift** option, the pipeline becomes flexible. If a new column appears in the source, the pipeline will automatically include it in the destination and mark the values for that column as *null* for any existing, older records that lack that specific data.

# "How would you load multiple data files from REST APIs using a single copy activity?"

**Explanation and Solution:**
To achieve this using only one copy activity, the creator recommends using a **parameterized data set** approach. The solution involves the following steps:

* **Creating a Parameterized Data Set:** You create a parameter for the file name (e.g., `P_file_name`) because that is the only part of the URL path that changes for each file.
* **Configuring Dynamic Content:** In the connection settings of your data set, you use **Add Dynamic Content** to construct the URL, substituting the static path with the parameter variable.
* **Iteration with ForEach:** Since you need to ingest multiple files, you use a **ForEach** activity. Within the *ForEach* settings, you define an array of the specific file names you want to ingest.
* **Encapsulation:** In the *Copy Activity* inside the *ForEach* loop, you map the *ForEach* item to the data set parameter using the expression `item().name` to dynamically fetch each file.

This method allows you to process multiple files dynamically without needing to create separate copy activities or data sets for each file.

# "How would you design a fault tolerant ADF pipeline to process billions of records daily?"

**Explanation and Solution:**
The creator emphasizes that there isn't just one solution; instead, you must combine several strategies to ensure resilience:

* **Retry Policy:** You should apply a retry policy to activities so that if a failure occurs due to transient network issues, the pipeline automatically attempts to re-run the task.
* **Leverage Parallelism:** When using *Mapping Data Flows*, you should utilize parallelism to partition data into smaller chunks, which enhances processing speed and efficiency for large datasets.
* **Alerting and Monitoring:** To stay informed, you can configure alert rules in the **Monitor** tab. You can set thresholds (e.g., if a pipeline fails after all retries, trigger an alert) to notify stakeholders via email or SMS, moving away from older methods like using *Logic Apps*.

# Your ADF pipeline is running slow, how would you identify and fix the performance bottlenecks?

**The Recommended Approach:**
1. **Monitor and Analyze:** Do not jump to conclusions. Start by monitoring the pipeline performance in the *Monitor* tab to identify exactly which activity is creating the bottleneck.
2. **Potential Causes:** Once the bottleneck is identified, investigate common culprits based on the activity type:
   * **Mapping Data Flows:** Consider **pruning the data** as early as possible in the initial stages of the pipeline rather than at the end to improve efficiency.
   * **Copy Activity:** If pulling data from on-premises to *Azure*, ensure the **self-hosted Integration Runtime** is not throttled or hitting its resource limits.

# "How would you implement incremental data load in Azure Data Factory when working with large transactional databases or let's say just large databases?"

**Explanation and Solution:**

1. **Watermark Tables:** This approach involves creating a table that stores a watermark (usually a timestamp or a high-water mark ID). During each pipeline run, the process reads the watermark value from this table and filters the source data to pull only the records updated after that specific date or ID. After the data load, the watermark table is updated with the latest value for future runs.

2. **Change Data Capture (CDC):** This is a feature within Azure Data Factory that automates the tracking of data changes. It is particularly effective for on-premises SQL Server databases. Unlike the watermark approach, which requires manual maintenance, CDC natively tracks changes and loads only the modified records into the destination, significantly simplifying the process.


# **"You have a JSON file with an unknown structure, how would you process it in Azure Data Factory?"**

**The Solution and Explanation:**
To process JSON data without a fixed schema, the creator recommends using **Mapping Data Flows** combined with the following strategies:

* **Schema Drift:** You should enable the **Schema Drift** option in your *Source* transformation. This allows the pipeline to handle dynamic or evolving JSON structures without failing when it encounters new keys or columns.
* **Debug Mode:** By turning on **Debug Mode**, you can spin up a temporary cluster to perform real-time data inspection. This lets you preview the JSON data and its nested structures before finalizing the pipeline logic.
* **Flatten Transformation:** Since JSON files often contain nested dictionaries or arrays, you can use the **Flatten** transformation within the data flow. This allows you to unpack those nested elements into standard tabular columns, making the data easier to process and load into your destination.



# **"How would you implement a real-time streaming pipeline in Azure Data Factory using Azure Data Factory only?"**

**Explanation and Solution:**

The recommended solution is to use an **Event-based Trigger** (also known as an **Event Grid trigger**):

* **Mechanism:** This trigger monitors your *Data Lake* storage. As soon as a new file arrives in the specified container, it automatically initiates the pipeline.
* **Administrative Prerequisite:** The creator notes a critical "real-world" detail: you must go to your Azure subscriptions and **register for Event Grid services**. This is no longer enabled by default, and failing to register these services will prevent the trigger from working, even if you configure the pipeline correctly.

# **"How would you load data to different locations based on the file name?"**

**Solution and Explanation:**

The creator outlines a robust, dynamic approach to solving this, avoiding hard-coded logic. The strategy involves the following steps:

*   **Retrieve File Metadata:** Use the **Get Metadata** activity to fetch the list of child items (files) from the source directory.
*   **Iterate with For Each:** Pass the output of the metadata activity into a **For Each** activity to loop through every file individually.
*   **Route with Switch Activity:** Inside the loop, use a **Switch** activity. This acts similarly to a *case statement* in programming languages.
*   **Conditional Logic:** Set the switch expression to `item().name`. Within the switch, create cases (e.g., one for "sales" and one for "products"). If the file name matches a specific string, the logic branches to the corresponding **Copy Activity**.
*   **Dynamic Destinations:** Within each branch, configure the **Copy Activity** to use parameters for the target folder path, ensuring files are directed to their correct destination containers or subdirectories dynamically.

# **"How would you store all the file names in a variable?"** and **"How can we just get the number of files stored in a folder?"**.

**Solution and Explanation:**

1. **Get Metadata Activity:** Use this activity to pull the list of `child items` from the specified folder.
2. **For Each Activity:** Connect the output of the metadata activity to a **For Each** loop to iterate through every individual file found.
3. **Append Variable Activity:** Inside the loop, use an **Append Variable** activity to add each file's name to an array-type variable.
4. **Get File Count:** Once the loop completes, use a **Set Variable** activity outside the loop. Set its value using the `length()` function on the array variable (e.g., `length(variables('fileNames'))`). This dynamically calculates the total number of items stored in the array.

# Question
# Write a SQL query using window functions to calculate a rolling 7-day average of campaign impressions.

Suppose:

campaign_id | date       | impressions
------------|------------|------------
C1          | 2026-09-01 | 100
C1          | 2026-09-02 | 150
C1          | 2026-09-03 | 200
...
Answer
If there is exactly one row per day:

**SELECT
    campaign_id,
    date,
    impressions,
    AVG(impressions) OVER (
        PARTITION BY campaign_id
        ORDER BY date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7_day_avg
FROM campaign_metrics;**

**Important follow-up**

If there can be missing dates, ROWS BETWEEN 6 PRECEDING doesn’t necessarily mean the previous six calendar days.

In that situation I’d first create a calendar/date spine and join the campaign data to it, or use a date-based RANGE where supported.

**Interview answer**

“I would first clarify whether there is exactly one record per calendar day. If yes, ROWS is straightforward. If dates can be missing, I would use a calendar table because six previous rows aren’t necessarily six previous days.”

That extra sentence demonstrates senior-level thinking.

**Question
How do you handle recursive queries to model hierarchical data?**

Example:

CEO


 ├── Director A

 │     ├── Manager A

 │     └── Manager B

 └── Director B

Table:

employee_id,
employee_name,
manager_id

Azure SQL approach

**Use a recursive CTE:**


> WITH EmployeeHierarchy AS
> (
>     SELECT
>         employee_id,
>         employee_name,
>         manager_id,
>         0 AS level
>     FROM employees
>     WHERE manager_id IS NULL
> 
>     UNION ALL
> 
>     SELECT
>         e.employee_id,
>         e.employee_name,
>         e.manager_id,
>         h.level + 1
>     FROM employees e
>     INNER JOIN EmployeeHierarchy h
>         ON e.manager_id = h.employee_id
> )
> SELECT *
> FROM EmployeeHierarchy
> OPTION (MAXRECURSION 100);

# 4. Optimizing a slow Azure Synapse query

**Step 1 — Understand the query plan**

Look for:

scans,
large shuffles,
data movement,
skew,
expensive joins,
spills

**Step 2 — Check table design**

For dedicated SQL pools:

distribution strategy
distribution key
replicated tables
partitioning
statistics

**Step 3 — Reduce data early**

Instead of:

SELECT *
FROM huge_table
JOIN another_table
filter before joining:

SELECT ...
FROM (
    SELECT ...
    FROM huge_table
    WHERE business_date >= '2026-09-01'
) a
JOIN ...

**Step 4 — Optimize joins**

Consider:

correct distribution
replicated small dimensions
avoiding unnecessary data movement

**Step 5 — Update statistics**

Outdated statistics can lead to bad execution plans.

**Step 6 — Review concurrency**

If many workloads are running simultaneously, workload management and resource allocation matter.

# 5. How do you handle schema drift?
**Example**

**Yesterday:**

customer_id,
name,
age

**Today:**

customer_id
,name
,age
,email

My approach
I distinguish between:

Expected schema evolution

and

Unexpected schema corruption.

Architecture:

![image_1789440430724.png](./image_1789440430724.png "image_1789440430724.png")

For ADF Mapping Data Flows, schema drift capabilities can help dynamically handle columns.


For a production data platform, I would additionally maintain:

schema version,
metadata,
data contract,
validation rules,
audit logs

# 6. Best practices for monitoring/debugging ADF

![image_1789440655366.png](./image_1789440655366.png "image_1789440655366.png")

# **7. On-premise data warehouse → Azure**

## **Interview Question:**
"You're tasked with migrating an on-premise data warehouse to Azure. What steps would you take and which services would you use?"

---

## **Executive Summary:**

Migrate on-prem DW to **Azure Synapse Analytics** (core DW) + **Azure Data Lake Gen2** (storage layers: Bronze/Silver/Gold) + **Azure Data Factory** (orchestration). Follow a **10-step framework**: assess current state → choose services → design architecture → plan migration approach → migrate ETL → refactor code → test → secure → cutover → optimize. Use **phased migration** with CDC for minimal downtime. Typical timeline: **6-7 months** for enterprise scale (50TB, 200 tables).

---

## **Quick Answer Framework:**

### **1. Assessment & Planning**
* Document inventory: sources, schemas, ETL jobs, dependencies, data volumes
* Measure growth rate, performance baseline, SLAs
* Map downstream consumers (BI, apps, APIs)

### **2. Choose Azure Services**
* **Azure Data Lake Gen2** (Bronze/Silver/Gold data layers)
* **Azure Data Factory** (ETL orchestration)
* **Azure Databricks** (optional, for ML + complex transformations)

### **3. Architecture**
```
On-Prem → ADF Ingestion → ADLS Bronze → Transformation → ADLS Silver → 
Business Logic → Synapse (Gold) → Power BI → End Users
```

### **4. Migration Strategy**
* **Big Bang:** All-at-once (small DW, tolerable downtime)
* **Phased:** Table-by-table (large DW, minimal downtime) ✅ Recommended
* **Transfer:** Azure Data Box (>40TB bulk), ADF Copy Activity (smaller), CDC (ongoing sync)

### **5. ETL Migration**
* Lift-and-shift SSIS packages to **Azure-SSIS Integration Runtime**
* Modernize to ADF Mapping Data Flows
* Shift to **ELT paradigm** (load raw, transform in Synapse)

### **6. Code Refactoring**
* Optimize star schemas for Synapse
* **Distribution:** Hash (large facts), Replicated (small dims <2GB), Round-robin (staging)
* **Partitioning:** Date-based for query performance
* Remove cursors → set-based operations

### **7. Testing**
* **Data validation:** Row counts, quality checks, sampling
* **Performance:** Benchmark queries (on-prem vs Azure)
* **UAT:** Business user validation
* **DR:** Backup/restore, failover

### **8. Security**
* Azure AD authentication, RBAC authorization
* TDE (at rest), SSL/TLS (in transit)
* Private endpoints, VNet integration, ExpressRoute
* **Microsoft Purview** for data governance

### **9. Cutover**
1. Final CDC sync
2. Freeze source (read-only)
3. Final validation
4. Switch connection strings
5. Monitor 24-48hrs
6. Keep rollback plan

### **10. Post-Migration**
* Performance tuning (execution plans, statistics, distribution adjustments)
* Cost optimization (pause/resume, lifecycle policies, reserved capacity)
* Monitoring setup (Azure Monitor, alerts)
* Documentation & training

---

## **Key Interview Talking Points:**

**Why Synapse?** MPP architecture for analytics vs Azure SQL's OLTP focus.

**Minimize Downtime?** Phased migration + CDC + parallel run period.

**Cost Control?** Auto-pause dev environments, serverless pools for ad-hoc, lifecycle policies.

**Biggest Risk?** Data quality issues & poor distribution strategies → thorough testing critical.

**ELT vs ETL?** Modern cloud favors ELT—leverage Synapse compute power, maintain raw data audit trails.

---

## **Timeline Example (50TB, 200 tables):**
* Weeks 1-4: Assessment, design, POC
* Weeks 5-8: Setup & security
* Weeks 9-20: Schema + ETL migration + data load
* Weeks 21-26: Testing + parallel run
* Week 27: Cutover
* Week 28+: Optimization

**Total: 6-7 months**

In [0]:
-- What is a Fact Table?

-- A FACT TABLE is the central table in a star or snowflake schema in a data warehouse.
-- It stores quantitative data (metrics/measures) about business processes and events.

-- Key Characteristics:
-- 1. Contains MEASURES/METRICS (quantitative data): sales amount, quantity sold, revenue, profit, count, etc.
-- 2. Contains FOREIGN KEYS to dimension tables (customer_id, product_id, date_id, store_id, etc.)
-- 3. Typically has a COMPOSITE PRIMARY KEY made up of the foreign keys
-- 4. Usually the LARGEST table in the data warehouse (high volume of transactions)
-- 5. Grows CONTINUOUSLY as new transactions occur

-- Example: Sales Fact Table

-- Fact Table: fact_sales
CREATE TABLE fact_sales (
    sale_id BIGINT,                    -- Surrogate key (optional)
    date_id INT,                        -- FK to dim_date
    customer_id INT,                    -- FK to dim_customer
    product_id INT,                     -- FK to dim_product
    store_id INT,                       -- FK to dim_store
    
    -- MEASURES (quantitative metrics)
    quantity_sold INT,
    unit_price DECIMAL(10,2),
    total_sales_amount DECIMAL(12,2),
    discount_amount DECIMAL(10,2),
    profit DECIMAL(12,2),
    
    PRIMARY KEY (date_id, customer_id, product_id, store_id)
);

-- Example Query: Total sales by product category and month
SELECT 
    d.year,
    d.month,
    p.category,
    SUM(f.total_sales_amount) AS total_sales,
    SUM(f.quantity_sold) AS total_quantity,
    SUM(f.profit) AS total_profit
FROM fact_sales f
INNER JOIN dim_date d ON f.date_id = d.date_id
INNER JOIN dim_product p ON f.product_id = p.product_id
GROUP BY d.year, d.month, p.category
ORDER BY d.year, d.month, total_sales DESC;

-- Types of Fact Tables:

-- 1. TRANSACTION FACT TABLE: One row per transaction (most granular)
--    Example: Each individual sale transaction

-- 2. PERIODIC SNAPSHOT FACT TABLE: One row per period (daily, monthly)
--    Example: Account balance at end of each day

-- 3. ACCUMULATING SNAPSHOT FACT TABLE: One row per process lifecycle
--    Example: Order fulfillment (order date, ship date, delivery date)

-- Fact Table vs Dimension Table:
-- FACT: Stores "what happened" (metrics, measures, quantities)
-- DIMENSION: Stores "who, what, where, when, why" (descriptive attributes)

# 8. Reduce pipeline latency from 30 minutes → <5 minutes

I would investigate the entire pipeline, not one service.

Source

 ↓

ADF

 ↓

ADLS

 ↓

Databricks

 ↓

Target

**Source**

-use CDC/incremental extraction

-avoid full table scans

-parallelize extraction

-partition large queries

**ADLS:**
-avoid millions of tiny files

-use Parquet/Delta

-appropriate partitioning

**Databricks**
-optimize Spark

-avoid unnecessary shuffles

-broadcast small dimensions

-handle skew

-right-size cluster

-incremental processing

**Target**

-batch writes

-optimize indexes/distribution

-avoid row-by-row processing

# 10. Late-arriving dimensions

**Suppose a transaction arrives:**

customer_id = C123
but customer dimension hasn’t arrived yet.

**Options**

Use an unknown/default dimension member:

customer_key = -1
Load the fact.

**When the dimension arrives:**

C123 → actual customer_key
Then update the fact if business requirements require it.

**Strong answer**
“I would avoid rejecting the fact just because the dimension arrived late. I can temporarily associate it with an inferred or unknown member and reconcile it when the actual dimension arrives.”


# File Format Comparison: CSV vs Parquet vs Delta vs Avro

---

## **Quick Comparison**

| Format | Type | Best For | Key Feature |
|--------|------|----------|-------------|
| **CSV** | Text, row-based | Data exchange, small datasets | Human-readable, universal |
| **Parquet** | Binary, columnar | Analytics, archival | 90% compression, column pruning |
| **Delta** | Parquet + transaction log | Production data warehouses | ACID, time travel, MERGE |
| **Avro** | Binary, row-based | Streaming (Kafka), fast writes | Schema evolution, streaming |

---

## **When to Use Each:**

**CSV:** External exchange, landing zones, <100MB datasets  
**Parquet:** Read-heavy analytics, immutable storage, cost optimization  
**Delta:** Production workloads (default choice), updates/deletes, ACID requirements  
**Avro:** Kafka ingestion, write-heavy workloads, event logging

---

## **File Size Example (1GB CSV baseline):**
```
CSV:     1,000 MB
Avro:      400 MB (60% smaller)
Parquet:   100 MB (90% smaller)
Delta:     110 MB (89% smaller)
```

---

## **Key Differences:**

**CSV:** No schema, no ACID, slow for large data, no column pruning  
**Parquet:** Columnar (fast analytics), but immutable (no updates)  
**Delta:** Parquet performance + ACID + time travel + native MERGE/DELETE  
**Avro:** Row-based (fast writes), excellent for streaming, schema evolution

---

## **Databricks Best Practice (Medallion Architecture):**

```
Bronze (raw) → CSV/JSON/Avro
Silver (cleaned) → Delta
Gold (aggregated) → Delta
```

**Use Delta as your default for production lakehouse workloads.**

---

## **Interview Answer:**

> "**Delta is the default** for Databricks production workloads—it gives you Parquet's columnar performance plus ACID transactions, time travel, and native MERGE operations. Use **Avro** for streaming ingestion from Kafka, **CSV** only for landing zones or external exchange, and **Parquet** for cost-optimized archival when you don't need transactional features."


# 12. How do you monitor ADF and alert failures?
I would create alerts for:

- pipeline failure
- unusually long duration
- missed SLA
- IR failure
- repeated retries

I’d also maintain an audit table:

pipeline_name
,run_id
,start_time
,end_time
,status
,records_read
,records_written
,error_message

This is especially useful for operational reporting.

# Interview Question: What is the purpose of Log Analytics and Application Insights in Azure Data Engineering?

---

## BRIEF ANSWER:

Log Analytics and Application Insights are Azure monitoring services that provide observability for data pipelines and applications.

---

## LOG ANALYTICS:
* Centralized log repository for Azure resources (ADF, Databricks, Synapse, etc.)
* Query logs using KQL (Kusto Query Language)
* Set up alerts based on log patterns (pipeline failures, performance degradation)
* Used for operational monitoring and troubleshooting
* Example: Query all ADF pipeline failures in the last 24 hours

---

## APPLICATION INSIGHTS:
* Application Performance Monitoring (APM) tool
* Tracks performance metrics, dependencies, exceptions, and custom events
* Provides end-to-end transaction tracing
* Real-time dashboards and smart detection of anomalies
* Used for application-level monitoring (custom Python/Spark apps, APIs, web services)
* Example: Track execution time of a custom transformation function, detect slow API calls

---

## KEY USE CASES IN DATA ENGINEERING:

### 1. Pipeline Monitoring:
* Track ADF/Databricks pipeline runs, failures, durations
* Alert on SLA breaches or unexpected failures

### 2. Performance Troubleshooting:
* Identify bottlenecks in data processing
* Analyze query performance, resource utilization

### 3. Dependency Tracking:
* Map data lineage and dependencies across services
* Understand impact of failures

### 4. Cost Optimization:
* Identify long-running or inefficient pipelines
* Monitor resource consumption patterns

### 5. Proactive Alerting:
* Set up alerts for error rates, latency thresholds
* Prevent data quality issues by catching anomalies early

---

## EXAMPLE LOG ANALYTICS KQL QUERY:

**Query ADF pipeline failures in last 24 hours:**

```kql
ADFPipelineRun
| where TimeGenerated > ago(24h)
| where Status == "Failed"
| project TimeGenerated, PipelineName, RunId, ErrorMessage
| order by TimeGenerated desc
```

---

## INTEGRATION PATTERN:

```
Azure Data Factory → Diagnostic Settings → Log Analytics Workspace
Databricks → Diagnostic Logs → Log Analytics Workspace
Custom Apps → Application Insights SDK → Application Insights
Log Analytics + Application Insights → Azure Monitor Alerts → Notification (Email/Teams/PagerDuty)
```

---

## ONE-LINE SUMMARY:

> **"Log Analytics centralizes operational logs for querying and alerting across Azure data services, while Application Insights provides deep application performance monitoring and tracing for custom code."**

# You find duplicates in your target table even though source data is clean. How will you debug the issue?

## **Brief Answer:**

If the source is clean but the target has duplicates, the issue is in the **pipeline logic**:

### **Common Causes:**

1. **Pipeline ran multiple times** without deduplication
   - Check job/pipeline run history in system tables or ADF monitoring
   - Solution: Add `MERGE` or `INSERT OVERWRITE` instead of `INSERT`

2. **No primary key enforcement**
   - Delta tables don't enforce PKs automatically
   - Solution: Add `MERGE ON <key>` or dedup before insert

3. **Window/partition overlap**
   - Incremental loads with overlapping date ranges
   - Solution: Use `WHERE NOT EXISTS` or correct the incremental filter

4. **CDC/streaming duplicate events**
   - Kafka/Event Hub may deliver same message twice
   - Solution: Dedup on `<record_id, timestamp>` or use `dropDuplicatesWithinWatermark()`

5. **Fan-out join**
   - Joining to a dimension with duplicates
   - Solution: Check dimension table for dupes, add `DISTINCT` or fix join logic

### **Debug Steps:**

```sql
-- 1. Confirm duplicates in target
SELECT customer_id, COUNT(*) as cnt
FROM target_table
GROUP BY customer_id
HAVING COUNT(*) > 1;

-- 2. Check pipeline run history
SELECT * FROM system.lakeflow.pipeline_events
WHERE pipeline_id = '<your_pipeline>'
ORDER BY timestamp DESC;

-- 3. Compare source vs target counts
SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM source_table;
SELECT COUNT(*), COUNT(DISTINCT customer_id) FROM target_table;
```

### **Quick Fix:**

```sql
-- Remove duplicates (keep latest record)
CREATE OR REPLACE TABLE target_table_clean AS
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_time DESC) as rn
  FROM target_table
)
WHERE rn = 1;
```

### **Prevention:**

Use `MERGE` instead of `INSERT` for incremental loads:

```sql
MERGE INTO target_table t
USING source_table s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

# 12. Scale from 10M to 100M records
I would evaluate:

**Ingestion**
Incremental/CDC.

**Storage**
Partitioning and Parquet/Delta.

**Processing**
Spark parallelism.

**Joins**
Broadcast small tables.

**Data skew**
Salting/repartitioning where appropriate.

**Architecture**
Avoid unnecessary full reloads.

# 17. Managed Identity vs Service Principal

---

## **Quick Comparison**

| Aspect | Managed Identity | Service Principal |
|--------|------------------|-------------------|
| **What is it?** | Azure-managed identity (automatically managed credentials) | Application identity (manually managed credentials) |
| **Credential Management** | Fully automated by Azure (no passwords/secrets to manage) | Manual (you create and rotate secrets/certificates) |
| **Best For** | Azure resource-to-resource authentication | External systems, multi-cloud, automation requiring explicit credentials |
| **Scope** | Limited to Azure resources that support MI | Works anywhere (on-prem, other clouds, external apps) |
| **Security** | Most secure (no credential exposure risk) | Secure if managed properly (risk if secrets leaked) |
| **Lifespan** | Tied to resource lifecycle | Independent, persistent until deleted |
| **Use Cases** | VM → Key Vault, ADF → ADLS, Databricks → Unity Catalog | CI/CD pipelines, on-prem apps, third-party tools |

---

## **Managed Identity**

### **What is it?**
An identity **automatically created and managed by Azure** for your Azure resource (VM, ADF, Databricks, Function App, etc.). Azure handles all credential lifecycle management — no passwords, secrets, or certificates for you to manage.

### **Types:**

**1. System-Assigned Managed Identity:**
* Tied directly to a single Azure resource
* Created and deleted with the resource
* Cannot be shared across resources
* Example: Your Databricks workspace has its own system-assigned identity

**2. User-Assigned Managed Identity:**
* Standalone Azure resource
* Can be assigned to multiple resources
* Lifecycle independent of resources using it
* Example: One identity shared by ADF + Databricks + Synapse

### **When to Use:**
✅ Azure resource accessing another Azure resource  
✅ ADF pipeline reading from ADLS Gen2  
✅ Databricks connecting to Unity Catalog  
✅ VM accessing Key Vault  
✅ Function App writing to Cosmos DB  
✅ **Default choice when both resources are in Azure**  

### **Advantages:**
* **Zero credential management** — Azure rotates credentials automatically
* **Most secure** — no secrets to store, leak, or expire
* **Simplest** — enable and assign RBAC roles
* **Audit-friendly** — clear identity tied to resource

### **Limitations:**
* Only works for Azure resources that support Managed Identity
* Cannot be used for on-premises systems
* Cannot be used by external (non-Azure) applications

---

## **Service Principal**

### **What is it?**
An **application identity** in Azure Active Directory (Entra ID). Think of it as a "service account" for applications. You manually create and manage its credentials (client secret or certificate).

### **Components:**
* **Application (App Registration):** The identity definition in Entra ID
* **Client ID:** Unique identifier (like a username)
* **Client Secret or Certificate:** Password/authentication credential (you manage this)
* **Tenant ID:** Your Azure AD tenant

### **When to Use:**
✅ Automation scripts, CI/CD pipelines (GitHub Actions, Azure DevOps)  
✅ On-premises applications accessing Azure resources  
✅ Multi-cloud scenarios (AWS Lambda → Azure Storage)  
✅ Third-party tools (Terraform, Databricks CLI)  
✅ External systems that need Azure access  
✅ Scenarios requiring explicit credential storage (config files, key vaults)  

### **Advantages:**
* Works **anywhere** (on-prem, other clouds, external apps)
* Full control over credential lifecycle
* Can be used across multiple resources and environments
* Portable (not tied to a single Azure resource)

### **Limitations:**
* **You manage secrets** — must rotate, secure, and monitor
* **Security risk** if secrets are leaked or hardcoded
* **More overhead** — secret expiration, rotation policies, storage
---

# **Question:** "How would you secure sensitive connection strings and credentials in ADF?"

**Solution and Explanation:**
*   **Azure Key Vault:** The primary solution is to use *Azure Key Vault*. It is used to store secrets and sensitive information securely, allowing you to avoid hardcoding credentials in your pipelines.
*   **Managed Identities:** Using *Managed Identities* is crucial and highly recommended by *Microsoft*. Because *Azure* automatically creates a managed identity with the same name as your *ADF* instance, you can assign it roles (like *Key Vault Administrator* or *Contributor*) to grant secure access to resources like *Key Vault* or *Synapse* without needing manual credentials.

# **Exact Question:** "You need to copy millions of records from an on-prem SQL Server to Azure Data Lake daily; how would you optimize the copy activity to reduce execution time?"

**Solution and Explanation:**
The creator explains that you can optimize the *Copy Activity* by using the following techniques:
* **Enable Staging:** This acts as an intermediate storage step, which can significantly improve performance when moving large volumes of data.
* **Bulk Insert:** This is a specialized option available specifically when using *SQL Server* as a source, which optimizes the data transfer process for high-volume loads.

By leveraging either or both of these settings, the *Copy Activity* will run more efficiently, allowing the pipeline to handle millions of records daily without excessive latency.

# **Question:** How would you integrate Databricks with Azure Data Factory, and how would you handle it if the pipeline fails due to transient issues?

**Solution:**
*   **Integration:** You can integrate Databricks by using the **Notebook Activity** within your pipeline. You will need to create a linked service with Databricks, which requires providing your Databricks workspace information and access tokens (which are obtained from the Databricks settings). Once the linked service is configured, you can directly run your Databricks notebooks.
*   **Handling Failures:** If the notebook execution fails due to transient or random issues, you can apply a **retry policy** in the activity settings. Additionally, you can add a failure notification activity or configure alerts through the monitor tab to notify yourself or stakeholders if the pipeline fails after the retries are exhausted.

# **Question:** How would you move data from a REST API that paginates responses in Azure Data Factory?

**Solution:**
*   **Data Set Configuration:** To handle paginated responses, you should use the **REST API** connector type for your source dataset rather than a generic HTTP connection. This specific connector provides built-in capabilities to handle pagination.
*   **Pagination Settings:** Within the copy activity source settings, you can configure the **Pagination** feature. By specifying the appropriate settings (such as configuring the absolute URL or pagination rules), the *Azure Data Factory* pipeline will dynamically read subsequent pages of data until all records have been retrieved.

# **Question:** How would you delete only those files that are older than 10 days in Azure Data Factory?

**Solution:**
* **Initial Metadata Extraction:** First, use the *Get Metadata* activity on the folder to retrieve the list of *child items*. Connect this to a *ForEach* activity to iterate through each file.
* **File-Level Metadata:** Within the *ForEach* loop, add a second *Get Metadata* activity targeting the specific file (using `item().name` as the file path). Configure the *Field list* to retrieve the *Last modified* property for that individual file.
* **Conditional Logic:** Use an *If Condition* activity following the second *Get Metadata* call. Define an expression that compares the file's *Last modified* date against the current UTC time (e.g., using functions like `subtractFromTime` or date difference logic).
* **Deletion:** If the condition evaluates to true (meaning the file is older than 10 days), use the *Delete* activity within the *True* path of the *If Condition* to remove that specific file.

# **Question:** How would you copy only those files from the source that are not already available in the destination?

**Solution:**
* **Metadata Extraction:** Use two *Get Metadata* activities—one to list all files in the source folder and another to list all files in the destination folder. Configure both to retrieve the *Child items* field.
* **Variable Storage:** Use a *ForEach* activity to iterate through the source files. Inside the loop, store the file list from the destination in a variable or use it to perform a comparison.
* **Conditional Logic:** Use an *If Condition* activity within the *ForEach* loop. The expression should use the `contains` function to check if the current source file name exists in the list of destination files. 
* **Data Movement:** In the *False* branch of the *If Condition* (meaning the file is not in the destination), add a *Copy* activity to move the file to the destination folder. In the *True* branch, you can simply add a *Wait* activity or leave it empty, as no action is required for existing files.

# **Detailed Solution and Explanation:**

1. **Key Vault Configuration:** To create and manage secrets, you must first navigate to your *Key Vault* resource in the *Azure Portal*. Within the **Access Control (IAM)** tab, you must assign yourself the *Key Vault Administrator* role. Only with this permission can you generate new secrets. Once created, store your sensitive value (e.g., a connection string or password) as a secret.

2. **ADF Authentication:** For ADF to access the *Key Vault*, it needs specific permissions. Go to the *Access Control* section of your *Key Vault* and assign the *Key Vault Administrator* (or *Key Vault Contributor*) role to your *Data Factory's* system-assigned managed identity. This allows the ADF instance to authenticate securely without requiring external credentials.

3. **ADF Web Activity Setup:** To call the secret, create a pipeline in ADF and use a **Web Activity**. This activity requires the secret's unique **Secret Identifier URL** from the *Key Vault*. 
   * **Method:** Set the method to *GET*.
   * **Authentication:** Select *System-assigned managed identity*.
   * **Resource:** Provide the required Microsoft documentation-specific resource string needed to authenticate against the vault.

4. **Dynamic Implementation:** To avoid hardcoding, you can parameterize the process. Create pipeline parameters for the *Key Vault name* and the *Secret name*. Within the *Web Activity*, replace the static URL components with these parameters using **Dynamic Content** and expressions. Ensure the API version is explicitly included at the end of the URL string. Finally, the output of the *Web Activity* will contain the secret value, which can be extracted and stored in an ADF variable for subsequent use in the pipeline.

# **Question:** Your pipeline processes large data sets and sometimes runs out of memory; how would you avoid this?

**Explanation and Solution:** When dealing with high-volume data in **data flows**, running out of memory is a common challenge. To mitigate this issue, the creator suggests two primary strategies:

1. **Enable Partitioning:** Within the source options of your data flow, you should configure data partitioning. By setting the partitioning to single, multiple, or other available modes, you can distribute the data, allowing it to process in parallel and reducing the memory strain.
2. **Data Pruning:** Implement the practice of pruning or filtering your data as early as possible in the initial stages of the pipeline. By removing unnecessary records or columns before they move further downstream, you significantly lower the memory footprint required for the transformation processes.

# **Question:** Your pipeline runs daily, but sometimes the same records get processed twice; how would you ensure that the process only handles each record once?

**Explanation and Solution:** To handle the issue of duplicate processing in a file-based environment (specifically within *data flows* and *lake houses*), the creator emphasizes the need for an **upsert** pattern. 

* **Use Inline Data Sets:** You should utilize **inline data sets** in data flows to create a **Delta Lake** data set, as standard formats like *CSV*, *Parquet*, or *JSON* do not natively support upserts.
* **Define Unique Keys:** By defining a **Source key** or a primary key within the sink configuration, the system can perform an update on existing records and an insert for new records. This **update plus insert** logic ensures that even if the pipeline runs multiple times, the data remains consistent and duplicates are avoided.

# **Question:** Your Azure Data Factory pipeline must process files in a specific order (e.g., a header file followed by a data file); how would you enforce this?

**Explanation and Solution:** By default, when using a **Get Metadata** activity in conjunction with a **For Each** activity to iterate over files, Azure Data Factory processes the items in parallel to save time. However, when a strict sequence is required, you must disable this parallel behavior.

* **Sequential Processing:** To enforce the order, go to the settings of the **For Each** activity. You will find a checkbox option for **Sequential**. Checking this box ensures that the pipeline processes the files one by one in the specific order returned by the **Get Metadata** activity, rather than running them concurrently.

# **Question:** You have two different files with different schemas; how would you process both files using a single copy activity while maintaining strict mapping requirements?

**Explanation and Solution:** 

Standard auto-mapping in a copy activity will not work when you require strict schema validation, as enabling mapping typically locks the activity to a single schema. The creator explains that you can overcome this by using **Dynamic Column Mapping**:

* **Extract Mapping Code:** First, configure the copy activity for each file individually to generate the correct mapping configuration. You can access the underlying JSON code by clicking the code (two brackets) icon in the pipeline editor.
* **Copy and Format:** Extract the relevant mapping portion from the JSON code for each file and verify it using a JSON formatter tool to ensure the structure is correct.
* **Implement Dynamic Mapping:** In your main pipeline, instead of selecting a static mapping, you can configure the mapping property of the copy activity to use dynamic content. By injecting the corresponding JSON mapping configuration based on the file being processed (e.g., using parameters or dynamic expressions), you allow the single copy activity to switch between different schema definitions at runtime.
* **Strict Validation:** By providing these specific JSON mappings, the pipeline will strictly enforce the expected columns for each file type. If a file arrives with columns that do not match the provided dynamic mapping, the pipeline will correctly identify the mismatch and fail, satisfying the requirement for strict schema control.

## **The Scenario:**

## Build an ADF pipeline that automatically detects newly uploaded CSV files in Azure Blob Storage, validates them using Python (null checks, deduplication), routes them to `Processed/` or `Rejected/` folders based on validation results, and logs the status of each file.

---

## **Architecture Overview:**

```
Azure Blob Storage (Landing) 
    ↓ [Trigger: New File Arrival]
Copy to ADLS Gen2 (Staging)
    ↓ [ForEach File]
Python Validation (Databricks/Azure Function)
    ↓ [If-Condition: Validation Result]
    ├─ Success → Move to Processed/ folder
    └─ Failure → Move to Rejected/ folder
    ↓
Log File Status (to Azure SQL/Log Analytics)
```

---

## **Step-by-Step Pipeline Design:**

### **Step 1: File Detection - Storage Event Trigger**

**Approach:**
* Use a **Storage Event Trigger** to automatically detect new CSV files uploaded to Azure Blob Storage
* Configure the trigger to monitor a specific container/folder (e.g., `landing/`)

**Configuration:**
```
Trigger Type: Storage Event Trigger
Blob Container: landing/
Blob Path Begins With: (optional) data/
Blob Path Ends With: .csv
Event: Blob Created
```

**Alternative (if Event Trigger not allowed):**
* Use a **Tumbling Window Trigger** (every 5/15 minutes) with a **Get Metadata** activity to list new files
* Filter files by `LastModified` timestamp to identify newly uploaded files

---

### **Step 2: Copy Files to ADLS Gen2 Staging**

**Activity: Copy Activity**

**Source:**
* **Linked Service:** Azure Blob Storage
* **Dataset:** Parameterized dataset pointing to the landing folder
* **File Path:** Use trigger output `@triggerBody().folderPath` and `@triggerBody().fileName`

**Sink:**
* **Linked Service:** ADLS Gen2
* **Dataset:** Parameterized dataset pointing to staging folder
* **File Path:** `staging/@{pipeline().parameters.fileName}`

**Key Settings:**
* **Binary Copy:** For this step, use binary copy to preserve the exact file
* **Dynamic File Name:** Pass `@triggerBody().fileName` as a pipeline parameter

---

### **Step 3: Process Multiple Files - ForEach Loop**

**Activity: Get Metadata Activity**

First, list all files in the staging folder:

```
Activity: Get Metadata
Dataset: ADLS Gen2 Staging Folder
Field List: Child Items
```

**Activity: ForEach Loop**

Iterate through each file:

```
Activity: ForEach
Items: @activity('Get Metadata').output.childItems
Sequential: false (for parallel processing)
Batch Count: 10 (process 10 files at a time)
```

---

### **Step 4: Python Validation - Databricks Notebook Activity**

**Inside ForEach Loop:**

**Activity: Databricks Notebook Activity**

**Notebook Path:** `/Shared/validation_notebook`

**Base Parameters:**
```json
{
  "file_path": "@{item().name}",
  "container": "staging",
  "storage_account": "mystorageaccount"
}
```

**Python Validation Script (Databricks Notebook):**

```python
# Get parameters
dbutils.widgets.text("file_path", "")
dbutils.widgets.text("container", "")
dbutils.widgets.text("storage_account", "")

file_path = dbutils.widgets.get("file_path")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

# Read CSV from ADLS Gen2

df = spark.read.format('csv').option('inferSchema',True).option('header',True).load(f"abfss://{container}@{storage_account}.dfs.core.windows.net/{file_path}")

# Validation 1: Check mandatory columns are not null/empty
mandatory_columns = ["customer_id", "order_id", "date"]
validation_status = "SUCCESS"
validation_message = ""
total_count = df.count()
valid_col = []
in_valid_col =[]

for i in df.columns:
    blank_count = df.filter(col(i).isNull | col(i)=="").count()
    blank_per  = (blank_count/total_count) * 100
    
    if blank_per = 0:
        valid_col.append(i)

    else:
        in_valid_col.append(i)

df_valid = df.select(valid_col)
# deduplicating the dataframe on the basis of latest record.
df_valid = df_valid.withColumn('dedup', row_number().over(Window.partitionBy('customer_id').orderBy(col('updated_time').desc()))).filter(col('dedup')==1).drop('dedup')

df_invalid = df.select(in_valid_col)

# Validation 2: Remove duplicates

df_cleaned = df_valid.dropDuplicates()

duplicate_count = total_count - df_cleaned.count()
    
    if duplicate_count > 0:
        validation_message = f"Removed {duplicate_count} duplicate records"
    
    # Write cleaned data to processed folder
    df_cleaned.write.format('delta').mode('append').option('path','abfss://{container}@{storage_account}.dfs.core.windows.net/processed').save()

    # Write invalid data to rejected foler
    df_invalid.write.format('delta').mode('append').option('path','abfss://{container}@{storage_account}.dfs.core.windows.net/rejected').save()

# Return validation result
dbutils.notebook.exit(f"{validation_status}|{validation_message}")
```

---

### **Step 6: Logging - Stored Procedure Activity**

**Create a Log Table in Azure SQL Database:**

```sql
CREATE TABLE FileProcessingLog (
    LogID INT IDENTITY(1,1) PRIMARY KEY,
    FileName NVARCHAR(255),
    ProcessedDateTime DATETIME,
    ValidationStatus NVARCHAR(50),
    ValidationMessage NVARCHAR(1000),
    DestinationFolder NVARCHAR(100),
    PipelineRunID NVARCHAR(100)
)
```

**Activity: Stored Procedure Activity**

**Stored Procedure Name:** `sp_LogFileStatus`

**Parameters:**
```json
{
  "FileName": "@{item().name}",
  "ProcessedDateTime": "@{utcNow()}",
  "ValidationStatus": "@{activity('Databricks Notebook').output.runOutput}",
  "ValidationMessage": "@{variables('ValidationMessage')}",
  "DestinationFolder": "@{if(startsWith(activity('Databricks Notebook').output.runOutput, 'SUCCESS'), 'processed', 'rejected')}",
  "PipelineRunID": "@{pipeline().RunId}"
}
```

**Stored Procedure:**
```sql
CREATE PROCEDURE sp_LogFileStatus
    @FileName NVARCHAR(255),
    @ProcessedDateTime DATETIME,
    @ValidationStatus NVARCHAR(50),
    @ValidationMessage NVARCHAR(1000),
    @DestinationFolder NVARCHAR(100),
    @PipelineRunID NVARCHAR(100)
AS
BEGIN
    INSERT INTO FileProcessingLog 
    VALUES (@FileName, @ProcessedDateTime, @ValidationStatus, @ValidationMessage, @DestinationFolder, @PipelineRunID)
END
```

---

## **Complete Pipeline Flow:**

```
1. Storage Event Trigger (detects new .csv in Blob landing/)
   ↓
2. Copy Activity: Blob → ADLS Gen2 staging/
   ↓
3. Get Metadata: List all files in staging/
   ↓
4. ForEach Loop: Iterate through each file
   ├─ 4.1: Databricks Notebook (Python Validation)
   ├─ 4.2: If Condition (Check validation result)
   │   ├─ TRUE: Copy to processed/ + Delete from staging/
   │   └─ FALSE: Copy to rejected/ + Delete from staging/
   └─ 4.3: Stored Procedure (Log status to Azure SQL)
```


## **Interview Talking Points:**

**1. Why Storage Event Trigger over Tumbling Window?**
* **Real-time processing:** Event trigger fires immediately when a file arrives
* **Cost-effective:** No unnecessary polling
* **Scalable:** Handles high-frequency file arrivals

**2. Why ForEach Loop?**
* **Batch processing:** Can handle multiple files uploaded simultaneously
* **Parallel execution:** Set `Sequential: false` to process files concurrently
* **Fault tolerance:** One file failure doesn't stop others

**3. Why Databricks over Data Flows?**
* **Complex validation logic:** Python provides flexibility for custom business rules
* **Reusable:** Same notebook can be called from multiple pipelines
* **Performance:** Spark handles large files efficiently

**4. Error Handling Best Practices:**
* **Retry Policy:** Set retry count to 3 for transient failures
* **Secure Output:** Enable for activities handling sensitive data
* **Dead Letter Queue:** Move failed files to a separate `failed/` folder after max retries
* **Alerts:** Set up Azure Monitor alerts for pipeline failures

**5. Performance Optimization:**
* **Parallel Processing:** Use ForEach with `Sequential: false` and `Batch Count`
* **Partition Files:** If files are large, partition before validation
* **Incremental Load:** Use `LastModified` metadata to avoid reprocessing

---

## **Bonus: Monitoring & Observability:**

* **Pipeline Run Monitoring:** Use ADF's built-in monitoring dashboard
* **Custom Logging:** Log to Azure SQL for detailed per-file tracking
* **Alerts:** Set up alerts for validation failure rate > threshold
* **Metrics:** Track processing time, file counts, failure rates

---

Would you like me to add this complete scenario to your notebook as a new cell?